# CS 6320 Project 2: Detecting Lines and Circles with the Hough Transform


**This notebook is your report.** Fill it in with code, figures, and written answers. Run every cell
before you submit so that the outputs are saved in the file - anything not visible in the submitted
notebook cannot be graded.

Total: **100 points**.


## Brief

- **Due:** See CANVAS
- **Hand in:** through Gradescope
- **Required file:** `<your_uid>.zip` (your UID starts with `u`, e.g. `u1234567.zip`)

## Overview

The Hough transform turns "find the shapes in this image" into "find the peaks in this array." Every
edge pixel votes for all the shapes that could pass through it, and shapes that many pixels agree on
show up as peaks in an accumulator. You will implement it twice - once for lines, once for circles -
and then watch it struggle with clutter, noise, and perspective.

- **Q1 (30 pts)** Smooth an image, detect edges, implement the line Hough transform, and draw both
  infinite lines and finite edge segments.
- **Q2 (20 pts)** Implement the circle Hough transform and find the coins.
- **Q3 (24 pts)** Run line detection on a cluttered image and deal with the false positives.
- **Q4 (26 pts)** Run circle detection on the same cluttered image.


## Setup

0. Unzip the project and enter its folder: `unzip proj2_6320.zip && cd proj2_6320`
1. Install [Miniconda](https://docs.conda.io/en/latest/miniconda.html) if you do not have conda.
2. Create the environment: `conda env create -f proj2_env.yml`. The file is at the top level of the
   project folder. On Windows use the "Anaconda Prompt"; on macOS/Linux any terminal works.
3. Activate it: `conda activate proj2`
4. Start Jupyter from the project folder (`jupyter notebook`) and open
   `proj2_code/Project_2_CS6320.ipynb`. The notebook reads and writes files using paths relative to
   `proj2_code/`, such as `../data/` and `../results/`, so leave it where it is.
5. Work through the notebook top to bottom. The setup cell creates the `results/Ques_1` through
   `results/Ques_5` folders for you.
6. Run every cell one last time, save, and build your submission with
   `python zip_submission.py --uid <your_uid>`.

## Library functions

**Do not use any library function that implements the Hough transform** — no `cv.HoughLines`,
`cv.HoughLinesP`, `cv.HoughCircles`, or `skimage.transform.hough_*`. The accumulator and the voting
have to be yours. Using the following functions is OK: `cv.filter2D` for convolution, `cv.line`,
`cv.circle`, and `cv.imwrite` for drawing and saving, and NumPy for anything you like.


## Debugging

`data/` includes three synthetic images (`synthetic-lines.jpg`, `synthetic-lines-1px.jpg`,
`synthetic-circles.jpg`). They are not graded, but if a detector misbehaves on a photograph it is
much faster to find out why on a picture with three clean lines in it.

`data/proj2-foreshortened-image.jpg` shows the same scene photographed at a steep angle. Nothing in
this project asks you to use it, but once everything else works it is worth ten minutes: run your
pipeline on it unchanged and see which coins survive and which do not. The answer is not random.

## Writeup

Do not change the order of the cells. You may add cells wherever you need them. Every image you save
into `results/` must also be displayed in the notebook - the `save_and_show()` helper in the setup
cell does both in one call, so use it.

## Rubric

| Question | Points | Breakdown |
| --- | --- | --- |
| Q1 Lines from a clean image | 30 | 1.1 Gaussian filter (3), 1.2 Sobel edges (6), 1.3 `hough_transform` (10), 1.4 draw lines (4), 1.5 draw edge segments (7) |
| Q2 Circles from a clean image | 20 | 2.1 `circle_hough_transform` (14), 2.2 draw circles (6) |
| Q3 Lines from a cluttered image | 24 | 3.1 smooth (3), 3.2 edges (3), 3.3 transform (3), 3.4 draw (3), 3.5 discussion (7), 3.6 pen boundaries (5) |
| Q4 Circles from a cluttered image | 26 | 4.1 smooth (3), 4.2 edges (3), 4.3 transform (3), 4.4 draw (3), 4.5 discussion (8), 4.6 at least 8 coins (6) |


**Q5 - required, 0 points.** Complete the AI-use disclosure at the end of the notebook,
including if you used no AI tools.

**Deductions.** −5 points each time the hand-in instructions are not followed (wrong file name,
missing outputs, absolute paths, notebook moved out of `proj2_code/`, missing AI-use
disclosure).

### Submission contents

`<your_uid>.zip`, produced by `zip_submission.py`, containing `proj2_code/` (your code and the run
notebook), `data/` (the input images), and `results/` (everything you generated). Please stay inside
the provided conda environment and use relative paths only.


In [1]:
import math
import os

import cv2 as cv
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

RESULTS = '../results'
for _q in range(1, 6):
    os.makedirs(f'{RESULTS}/Ques_{_q}', exist_ok=True)

np.random.seed(0)


def to_gray(bgr: np.ndarray) -> np.ndarray:
    """BGR (as returned by cv.imread) to a single-channel uint8 image."""
    if bgr.ndim == 2:
        return bgr
    return cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)


def show_bgr(img: np.ndarray, title: str = None, figsize=(6, 5)) -> None:
    """Display a BGR image with the channels put back in the right order."""
    plt.figure(figsize=figsize)
    plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    if title:
        plt.title(title, fontsize=10)
    plt.axis('off')
    plt.show()
    plt.close()


def show_gray(img: np.ndarray, title: str = None, figsize=(6, 5),
              vmin: float = 0, vmax: float = 255) -> None:
    """Display a single-channel image with the display range pinned to [vmin, vmax].

    Without vmin/vmax, matplotlib rescales the array to its own min and max. A binary
    image holding 0 and 1 would then look identical to one holding 0 and 255, and a
    nearly blank edge map would look full of detail.
    """
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
    if title:
        plt.title(title, fontsize=10)
    plt.axis('off')
    plt.show()
    plt.close()


def save_and_show(path: str, img: np.ndarray, title: str = None, gray: bool = None,
                  figsize=(6, 5)) -> None:
    """Write an image to disk and display it, checking that the write succeeded.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not cv.imwrite(path, img):
        raise IOError(f'cv.imwrite failed for {path} (dtype={img.dtype}, shape={img.shape})')
    if gray is None:
        gray = img.ndim == 2
    if gray:
        show_gray(img, title, figsize=figsize)
    else:
        show_bgr(img, title, figsize=figsize)


def plot_accumulator(accumulator, thetas, rhos, title, save_path, figsize=(20, 20)) -> None:
    """Display and save a line-Hough accumulator with real axis units."""
    plt.figure(figsize=figsize)
    plt.imshow(accumulator, cmap='hot', interpolation='none', aspect='auto',
               extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[-1], rhos[0]])
    plt.title(title, fontsize=16)
    plt.xlabel('theta (degrees)', fontsize=14)
    plt.ylabel('rho (pixels)', fontsize=14)
    plt.colorbar(label='votes')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    plt.close()


def edge_threshold_report(edges: np.ndarray, name: str) -> None:
    """Print how many pixels survive binarization at several thresholds.

    This is the number to look at when choosing a threshold: it is what determines how
    many votes the accumulator receives.
    """
    total = edges.size
    print(f'{name}: max={edges.max()}, mean={edges.mean():.1f}')
    for t in (60, 100, 140, 180, 220):
        n = int((edges > t).sum())
        print(f'    threshold {t:3d} -> {n:6d} edge pixels ({100 * n / total:5.2f}%)')


## Q1. Detecting lines in a clean image (30 points)

We will work with `data/proj2-clean-image.jpg`: two pens and a pile of coins on a dark background.
The pens give us lines, the coins give us circles in Q2.

### Tasks
1. Compute a smoothed version of the input image - **3 points**
2. Detect edges in the original and the smoothed image - **6 points**
3. Implement the line Hough transform - **10 points**
4. Draw the detected lines - **4 points**
5. Draw finite edge segments instead of infinite lines - **7 points**


### 1.1 Smooth the input image (3 points)

Write `Gaussian_Filter(sigma)`, which returns a square 2D Gaussian kernel.

- The only argument is `sigma`. The kernel **size** must be derived from it, not passed in: a
  Gaussian is effectively zero beyond about three standard deviations, so a common choice is
  `size = 2 * int(3 * sigma) + 1`, which is always odd and always contains the bulk of the mass.
- The kernel must be **normalized** so its entries sum to 1. If it is not, smoothing will change the
  overall brightness of the image.
- Return a `float32` array so `cv.filter2D` accepts it.


In [2]:
def Gaussian_Filter(sigma):
    """Build a normalized 2D Gaussian kernel from sigma alone.

    Args:
        sigma: standard deviation of the Gaussian, in pixels

    Returns:
        A square float32 array of odd side length whose entries sum to 1.
    """
    w = 2 * int(3 * sigma) + 1 # recommended half-width is at least 3-sigma 
    ax = np.linspace(-w + 1, w - 1, w) / 2
    gauss = np.exp((ax / sigma)**2 / 2)
    kernel = np.outer(gauss, gauss)
    return kernel / np.sum(kernel)

In [ ]:
img = cv.imread('../data/proj2-clean-image.jpg')
assert img is not None, 'could not read ../data/proj2-clean-image.jpg'
print('image shape:', img.shape)

SMOOTH_SIGMA = 2
g_kernel = Gaussian_Filter(SMOOTH_SIGMA)
print(f'kernel shape {g_kernel.shape}, sum {g_kernel.sum():.6f}, dtype {g_kernel.dtype}')

smooth_img = cv.filter2D(img, -1, g_kernel)

show_bgr(img, 'original')
save_and_show(f'{RESULTS}/Ques_1/proj2-clean-image-smoothed.png', smooth_img,
              f'smoothed, sigma = {SMOOTH_SIGMA}')


### 1.2 Edge detection (6 points)

The Hough transform votes with edge pixels, so we need edges first.

Write `detect_edges_with_sobel(image)`, which takes a **grayscale** image and returns the gradient
magnitude:

1. Convolve the image with the Sobel-x and Sobel-y kernels using `cv.filter2D`. You do not need to
   write your own convolution.
2. Combine the two gradients into a magnitude, `sqrt(gx**2 + gy**2)`.
3. Clip to [0, 255] and return `uint8`.

Two practical points. Pass `cv.CV_32F` as the `ddepth` argument to `cv.filter2D` - with the default
depth the intermediate gradients would be clipped to `uint8` and every negative gradient would be
destroyed. And always convert to grayscale first: running this on a 3-channel BGR array produces a
3-channel result that will quietly break everything downstream.

Apply it to **both** the original and the smoothed image.


In [ ]:
def detect_edges_with_sobel(image):
    """Gradient magnitude of a GRAYSCALE image, via the two Sobel kernels.

    Args:
        image: single-channel image

    Returns:
        uint8 gradient magnitude, clipped to [0, 255].
    """
    ############################
    ### TODO: Your Implementation ###
    raise NotImplementedError('detect_edges_with_sobel() needs to be implemented.')
    ############################
    ### Your Implementation ENDS here ###
    ############################

    return img_gray_edge


#### 1.2.1 Edges in the original image

Store the result in `img_gray_edge`.


In [ ]:
img_gray = to_gray(img)
img_gray_edge = detect_edges_with_sobel(img_gray)

save_and_show(f'{RESULTS}/Ques_1/Ori_Clean_Edge_Image.png', img_gray_edge,
              'edges, original image')
edge_threshold_report(img_gray_edge, 'original')


#### 1.2.2 Edges in the smoothed image

Store the result in `smooth_img_gray_edge`.


In [ ]:
############################
### TODO: Your Implementation ###
# Convert smooth_img to grayscale and run detect_edges_with_sobel on it.
# Store the result in smooth_img_gray_edge.
raise NotImplementedError('Edge detection on the smoothed image needs to be implemented.')
############################
### Your Implementation ENDS here ###
############################

save_and_show(f'{RESULTS}/Ques_1/Smooth_Clean_Edge_Image.png', smooth_img_gray_edge,
              'edges, smoothed image')
edge_threshold_report(smooth_img_gray_edge, 'smoothed')


#### 1.2.3 Compare them

Look at the two edge images and the printed statistics. What did smoothing do to the edge map, and
what would you expect that to do to the Hough transform downstream? Answer in the cell below.


#### Your answer to 1.2.3

*Two or three sentences comparing the two edge maps, and what you expect the difference to mean for
line detection.*


### 1.3 Implement the line Hough transform (10 points)

A line can be written as

$$\rho = x\cos\theta + y\sin\theta$$

where $\rho$ is the perpendicular distance from the origin to the line, and
$\theta$ is the angle of that perpendicular. Every edge pixel $(x, y)$ is consistent with a whole
family of lines - one for each $\theta$ - so it casts one vote per $\theta$ bin. Collinear pixels
all vote for the same $(\rho, \theta)$ cell, which is why real lines become peaks.

Write `hough_transform(input_img, threshold, theta_res, rho_res)`:

- **Binarize first.** Keep pixels whose edge magnitude is strictly greater than `threshold`. Return
  this binary image as `uint8` with values **0 and 255** (not 0 and 1) so that it can be saved and
  viewed directly.
- **$\theta$ ranges over [−90, 90) degrees** in steps of `theta_res`. With `theta_res = 1` that is
  −90, −89, …, 89. This half-open range is enough: $(\rho, \theta)$ and $(-\rho, \theta + 180°)$
  describe the same line.
- **$\rho$ ranges over [−d, d]** in steps of `rho_res`, where `d` is the image diagonal rounded up.
  Negative values are needed because $\rho$ is signed.
- Return `(accumulator, thetas, rhos, binary_img)`, where `accumulator[i, j]` is the vote count for
  $\rho$ = `rhos[i]` and $\theta$ = `thetas[j]`, and `thetas` is **in radians** (convert once, up
  front, so that the drawing functions can use it directly).

Map a computed $\rho$ to its bin with `round((rho - rhos[0]) / rho_res)` rather than
`np.searchsorted`, which puts values on a bin boundary in the wrong bin.

**Vectorize the inner loop.** Loop over `thetas` in Python, but compute $\rho$ for *all* edge pixels
at once as an array, then accumulate with `np.bincount`. The naive double loop is several hundred
times slower and you will be calling this function a lot in Q5.


In [ ]:
def hough_transform(input_img, threshold, theta_res, rho_res):
    """Vote for lines, using rho = x*cos(theta) + y*sin(theta).

    Args:
        input_img: edge magnitude image (grayscale)
        threshold: pixels with magnitude strictly greater than this vote
        theta_res: theta step in DEGREES
        rho_res:   rho step in pixels

    Returns:
        accumulator: (len(rhos), len(thetas)) integer vote counts
        thetas:      angles in RADIANS, covering [-90, 90) degrees
        rhos:        rho values, covering [-diagonal, diagonal]
        binary_img:  uint8 image with values 0 and 255
    """
    ############################
    ### TODO: Your Implementation ###
    raise NotImplementedError('hough_transform() needs to be implemented.')
    ############################
    ### Your Implementation ENDS here ###
    ############################

    return accumulator, thetas, rhos, binary_img


#### 1.3.1 Lines from the original image

Run the transform on the edges of the original image and plot the accumulator.

Choose the binarization `threshold` yourself. The setup cell printed how many pixels survive at
several thresholds, which is the useful thing to look at: too low and the accumulator fills with
votes from texture and noise, too high and the pen edges become dashed and their peaks fall below
everything else. **Report the value you chose and say why** in the answer cell for 1.3.


In [ ]:
############################
### TODO: choose the binarization threshold for the ORIGINAL image's edges.
ori_bin_threshold = None
############################

ori_accumulator, ori_thetas, ori_rhos, ori_binary_img = hough_transform(
    img_gray_edge, ori_bin_threshold, theta_res=1, rho_res=1)

print(f'accumulator {ori_accumulator.shape}, max votes {ori_accumulator.max()}, '
      f'{int((ori_binary_img > 0).sum())} edge pixels voted')

save_and_show(f'{RESULTS}/Ques_1/Ori_Clean_Binary_Image.png', ori_binary_img,
              f'binary edges, threshold = {ori_bin_threshold}')
plot_accumulator(ori_accumulator, ori_thetas, ori_rhos,
                 'Hough accumulator, original image',
                 f'{RESULTS}/Ques_1/Ori_Clean_Image_Accumulator_Array.png')


#### 1.3.2 Lines from the smoothed image

Same thing, using the edges of the smoothed image. The threshold that worked above will probably not
be right here - smoothing lowers gradient magnitudes across the board.


In [ ]:
############################
### TODO: choose the binarization threshold for the SMOOTHED image's edges,
###       then run the transform, save the binary image and plot the accumulator.
###       Name the outputs smooth_accumulator, smooth_thetas, smooth_rhos,
###       smooth_binary_img and save to the Smooth_Clean_* paths.
raise NotImplementedError('1.3.2 needs to be implemented.')
############################
### Your Implementation ENDS here ###
############################


#### Your answer to 1.3

*State the binarization threshold you used for each image and how you chose it. Looking at the two
accumulator plots, describe the effect of that threshold on the result: what happens when it is set
too low, and what happens when it is set too high?*


### 1.4 Draw the detected lines (4 points)

Write `plot_lines(img_path, accumulator, thetas, rhos, acc_threshold, save_path, nhood)`, which draws
every detected line across the full width of the image.

Two steps:

1. **Pick the peaks.** Thresholding the accumulator alone is not enough. A strong line does not
   produce a single hot cell but a small blob of them, so plain thresholding gives you a dozen
   near-identical lines stacked on top of each other. Keep only cells that are a local maximum within
   a `nhood = (rho_window, theta_window)` neighbourhood **and** above `acc_threshold`. Tip: you can use `cv.dilate`
   with a rectangular kernel which gives you the local maximum in one call: a cell is a local max wherever
   the dilated array equals the original.
2. **Draw each line.** `cv.line` needs two endpoints, so intersect the line with the image border.
   Solving $\rho = x\cos\theta + y\sin\theta$ for `x` at `y = 0` and `y = h-1` works for most
   lines but divides by zero for horizontal ones, so handle both cases: also solve for `y` at `x = 0`
   and `x = w-1`, collect the valid intersections, and use the two that are farthest apart.

A sensible starting point is `nhood = (15, 5)` and an `acc_threshold` around 60% of the accumulator
maximum, but tune it and say what you used.


In [ ]:
def find_line_peaks(accumulator, acc_threshold, nhood=(15, 5)):
    """Accumulator cells that are above threshold AND a local maximum.

    Args:
        accumulator: 2D vote array
        acc_threshold: minimum vote count
        nhood: (rho_half_window, theta_half_window) for the local-maximum test

    Returns:
        (N, 2) array of (rho_index, theta_index), strongest first.
    """
    ############################
    ### TODO: Your Implementation ###
    raise NotImplementedError('find_line_peaks() needs to be implemented.')
    ############################
    ### Your Implementation ENDS here ###
    ############################


def line_endpoints(rho, theta, height, width):
    """Where the line (rho, theta) crosses the image border.

    Returns:
        ((x0, y0), (x1, y1)) for the two most distant intersections, or None.
    """
    ############################
    ### TODO: Your Implementation ###
    raise NotImplementedError('line_endpoints() needs to be implemented.')
    ############################
    ### Your Implementation ENDS here ###
    ############################


def plot_lines(img_path, accumulator, thetas, rhos, acc_threshold, save_path, nhood=(15, 5)):
    """Draw every detected line across the image and save the result.

    Returns:
        The list of (rho, theta) pairs that were drawn.
    """
    ############################
    ### TODO: Your Implementation ###
    raise NotImplementedError('plot_lines() needs to be implemented.')
    ############################
    ### Your Implementation ENDS here ###
    ############################


In [ ]:
############################
### TODO: choose the accumulator threshold and draw the lines on BOTH images.
###       Save to Ori_Clean_Image_Output_Lines.png and
###       Smooth_Clean_Image_Output_Lines.png in results/Ques_1.
raise NotImplementedError('1.4 needs to be implemented.')
############################


### 1.5 Draw edge segments instead of infinite lines (7 points)

The lines from 1.4 run off the edge of the image. What we actually want is the part of each line
where the pen really is.

Write `plot_edges(img_path, accumulator, thetas, rhos, acc_threshold, binary_image, save_path, nhood)`.
For each peak you selected, find the edge pixels that lie on that line and draw a segment spanning them. 

Tip: Find the edge pixels that voted for the line or the ones whose distance
$|x\cos\theta + y\sin\theta - \rho|$ is below a small tolerance, and use the extreme projected points as candidate endpoints. 



In [ ]:
def plot_edges(img_path, accumulator, thetas, rhos, acc_threshold, binary_image,
               save_path, nhood=(15, 5), tol=2.0):
    """Draw finite segments covering the edge pixels that lie on each detected line.

    Args:
        binary_image: the binary edge image the accumulator was built from
        tol: how close (in pixels) an edge pixel must be to count as on the line

    Returns:
        The list of ((x0, y0), (x1, y1)) segments drawn.
    """
    ############################
    # TO DO: Your Implementation
    raise NotImplementedError('plot_edges() needs to be implemented.')
    ############################
    # Your Implementation ENDS here
    ############################


In [ ]:
############################
### TODO: draw the edge segments on BOTH images, saving to
###       Ori_Clean_Image_Output_Edges.png and Smooth_Clean_Image_Output_Edges.png
raise NotImplementedError('1.5 needs to be implemented.')
############################


## Q2. Detecting circles in a clean image (20 points)

A circle needs three parameters - center $(a, b)$ and radius $r$ - so the accumulator is
three-dimensional. An edge pixel at $(x, y)$ is on a circle of radius $r$ whenever the center sits at

$$a = x - r\cos\phi, \qquad b = y - r\sin\phi$$

for some angle $\phi$. So each edge pixel votes for a whole ring of candidate centers at each
radius.

### Tasks
1. Implement `circle_hough_transform` - **14 points**
2. Draw the detected circles on the original and the smoothed image - **6 points**


### 2.1 Implement the circle Hough transform (14 points)

Write `circle_hough_transform(input_img, threshold, min_radius, max_radius, radius_res, angle_res)`.

- Binarize the edge image exactly as in `hough_transform` (values 0 and 255).
- `radius_vector = arange(min_radius, max_radius + 1, radius_res)`.
- **`angle_res` is in degrees**, and $\phi$ sweeps the full circle, `arange(0, 360, angle_res)`.
  Five degrees (72 votes per pixel per radius) is plenty; using radians here by mistake gives you
  about six votes per ring and the accumulator will look strangely flat.
- Return `(accumulator, radius_vector, center_x_vector, center_y_vector, binary_img)` where
  `accumulator[y, x, k]` holds the votes for a circle centered at `(x, y)` with radius
  `radius_vector[k]`. `center_x_vector` and `center_y_vector` are just `arange(width)` and
  `arange(height)` - the coordinate axes the accumulator is indexed by.

Index the accumulator by **radius index** `k`, not by the radius value. Indexing by value wastes
memory and silently leaves gaps whenever `radius_res > 1`.

Vectorize over edge pixels the same way as before: loop over radii and angles. Votes that land outside the image must be discarded. 

Tip: you may use `np.bincount` on flattened `(b * width + a)` indices for all edge pixels at once to accelerate voting.

**Choosing the radius range matters more than you would think.** Wider is not better: every extra radius adds another full pass over the edge pixels, and radii that match nothing still collect background votes. Measure a few coins in an image viewer and bracket what you find.


In [ ]:
def circle_hough_transform(input_img, threshold, min_radius, max_radius,
                          radius_res=1, angle_res=5):
    """Vote for circles: a = x - r*cos(phi), b = y - r*sin(phi).

    Args:
        input_img: edge magnitude image
        threshold: binarization threshold
        min_radius, max_radius: inclusive radius range, in pixels
        radius_res: radius step
        angle_res: phi step in DEGREES

    Returns:
        accumulator: (height, width, n_radii) votes, indexed [y, x, radius_index]
        radius_vector, center_x_vector, center_y_vector: the axes of the accumulator
        binary_img: uint8 image with values 0 and 255
    """
    ############################
    # TO DO: Your Implementation
    raise NotImplementedError('circle_hough_transform() needs to be implemented.')
    ############################
    # Your Implementation ENDS here
    ############################

    return accumulator, radius_vector, center_x_vector, center_y_vector, binary_img


#### 2.2.1 Circles in the original image

You should be able to find **at least 7 coins** to get 75% of the grade. Report how many you found.


In [ ]:
############################
### TODO: choose the binarization threshold and radius range, then run the transform
###       on the ORIGINAL image's edges. Save the binary image to
###       Ori_Clean_Binary_Image_Circle_Detection.png in results/Ques_2.
raise NotImplementedError('2.1.1 needs to be implemented.')
############################


#### 2.2.2 Circles in the smoothed image

Same thing on the smoothed image. If you use a different sigma or a different binarization threshold
here than you did for line detection, say so and explain why in the answer cell. You should be able to find **at least 7 coins** to get 75% of the grade


In [ ]:
############################
### TODO: same on the SMOOTHED image's edges. Name the outputs
###       smooth_circle_accumulator, smooth_circle_radius_vector,
###       smooth_circle_center_x_vector, smooth_circle_center_y_vector,
###       smooth_circle_img_bin and save the binary image to
###       Smooth_Clean_Binary_Image_Circle_Detection.png
raise NotImplementedError('2.1.2 needs to be implemented.')
############################


### 2.2.3 Draw the circles (6 points)

Write `plot_circles(img_path, accumulator, radius_vector, center_x_vector, center_y_vector,
acc_threshold, min_center_dist, binary_image, save_path)`.

The same peak problem as in 1.4 appears here, and worse: a single coin produces a cloud of high-vote
cells at neighbouring centers and neighbouring radii, so thresholding alone draws twenty concentric
rings on each coin. Do a greedy **non-maximum suppression** instead:

1. Collect every `(x, y, k)` above `acc_threshold`.
2. Sort them by vote count, highest first.
3. Walk down the list and keep a candidate only if its center is at least `min_center_dist` pixels
   away from every center already kept.

Setting `min_center_dist` to roughly the smallest coin radius works well. Draw each kept circle with
`cv.circle`, and return the list so the cell can report how many were found.


In [ ]:
def plot_circles(img_path, accumulator, radius_vector, center_x_vector, center_y_vector,
                 acc_threshold, min_center_dist, binary_image, img_save_path):
    """Select circles by threshold + non-maximum suppression, draw them, and save.

    Returns:
        List of (x, y, radius, votes), strongest first.
    """
    ############################
    # TO DO: Your Implementation
    raise NotImplementedError('plot_circles() needs to be implemented.')
    ############################
    # Your Implementation ENDS here
    ############################


In [ ]:
############################
### TODO: choose acc_threshold and min_center_dist, then draw the circles on BOTH
###       images (Ori_Clean_Image_Output_Circles.png and
###       Smooth_Clean_Image_Output_Circles.png) and report how many were found.
raise NotImplementedError('2.2 needs to be implemented.')
############################


#### Your answer to 2.2

*How many coins did each run find, and how many are actually in the image? Report the parameters you
settled on (binarization threshold, radius range, accumulator threshold, minimum center distance). If you use a different sigma or a different binarization threshold
here than you did for line detection, say so and explain why in your answer*


## Q3. Lines in a cluttered image (24 points)

`data/proj2-clutter-image.jpg` has the same pens and coins, but now on a patterned book cover with a
printed card underneath. Everything you wrote in Q1 still applies; the question is whether it still
works.

### Tasks
1. Smooth the image and comment on the effect -  **3 points**
2. Detect edges - **3 points**
3. Run the line Hough transform - **3 points**
4. Draw the detected lines - **3 points**
5. Explain why most of those lines are not pen boundaries - **7 points**
6. Recover the pen boundaries - **5 points**


### 3.1 Smooth the cluttered image (3 points)

Pick a sigma, smooth the image, and save the result as `clutter_smooth_img`.

Do not just reuse the value from Q1 without checking. The background here is high-contrast print, and
the pens are thin. Try at least two values and look at what happens to both the clutter and the pen
edges before committing. Report your choice and the reasoning in the 3.5 answer cell.


In [ ]:
clutter_img = cv.imread('../data/proj2-clutter-image.jpg')
assert clutter_img is not None, 'could not read ../data/proj2-clutter-image.jpg'

############################
# TO DO: Your Implementation
# Choose a sigma, smooth clutter_img, and store the result in clutter_smooth_img.
raise NotImplementedError('3.1 needs to be implemented.')
############################
# Your Implementation ENDS here
############################

save_and_show(f'{RESULTS}/Ques_3/clutter_smooth_img.png', clutter_smooth_img,
              'smoothed cluttered image')


### 3.2 Detect edges (3 points)

Reuse `detect_edges_with_sobel`. Save the result as `clutter_smooth_img_gray_edge`. Remember to
convert to grayscale first.


In [ ]:
############################
# TO DO: Your Implementation
# Store the edges in clutter_smooth_img_gray_edge.
raise NotImplementedError('3.2 needs to be implemented.')
############################
# Your Implementation ENDS here
############################

save_and_show(f'{RESULTS}/Ques_3/clutter_smooth_img_gray_edge.png',
              clutter_smooth_img_gray_edge, 'edges, smoothed cluttered image')
edge_threshold_report(clutter_smooth_img_gray_edge, 'clutter (smoothed)')


### 3.3 Run the line Hough transform (3 points)

In [ ]:
############################
# TO DO: Your Implementation
# Run hough_transform on clutter_smooth_img_gray_edge. Name the outputs
# clutter_smooth_accumulator, clutter_smooth_thetas, clutter_smooth_rhos,
# clutter_smooth_binary_img, and save the binary image to
# results/Ques_3/Clutter_Smooth_Binary_Image.png
raise NotImplementedError('3.3 needs to be implemented.')
############################
# Your Implementation ENDS here
############################


### 3.4 Draw the detected lines (3 points)

Draw them on the **smoothed color image**, not on the binary edge image — you need to be able to see
what the lines are landing on.


In [ ]:
############################
# TO DO: Your Implementation
# Draw the lines on the SMOOTHED COLOR image and save to
# results/Ques_3/Clutter_Smooth_Image_Output_Lines.png
raise NotImplementedError('3.4 needs to be implemented.')
############################
# Your Implementation ENDS here
############################


### 3.5 Why are most of those lines not pen boundaries? (7 points)

Look at the accumulator and at the drawn result, then answer in the cell below. Useful things to
check before you write: what angles do the strongest peaks sit at, and what in the photograph is
oriented that way? Is the problem that the accumulator is wrong, or that it is right about something
you are not interested in?


#### Your answer to 3.5

*What sigma did you choose in 3.1 and why? What do the strongest accumulator peaks correspond to in
this photograph? Can you separate pens from clutter by raising the accumulator threshold alone, and
if not, what extra information would you need?*


### 3.6 Recover the pen boundaries (5 points)

Write a function like `plot_edges` but with a length filter - call it `plot_line_segments` so that `plot_edges` still works if you re-run an earlier cell.

The idea: a line that the accumulator likes may still be made of scattered pixels. Gather the edge pixels on each candidate line, sort them along the line as in 1.5, then **split them into runs** wherever the gap between consecutive pixels exceeds `max_gap`. Each run is a candidate segment. Keep only runs whose length falls inside `[min_length, max_length]`.

Save your result as `../results/Ques_3/Clutter_Smooth_Image_Output_Edges.png`.

Note: Recovering both pens cleanly with this pipeline is hard. If you find one pen, then provide a diagnosis of why the other is missing.


In [ ]:
def plot_line_segments(img_path, accumulator, thetas, rhos, acc_threshold, binary_image,
                       save_path, nhood=(15, 5), tol=2.0, max_gap=30,
                       min_length=0, max_length=np.inf):
    """Like plot_edges, but splits each line into runs and filters them by length.

    Args:
        max_gap: a gap larger than this between consecutive pixels starts a new run
        min_length, max_length: keep only runs whose length falls in this band

    Returns:
        List of (p0, p1, length, theta_degrees), longest first.
    """
    ##############################
    # TO DO: Your Implementation
    raise NotImplementedError('plot_line_segments() needs to be implemented.')
    ################################
    # Your Implementation ENDS here
    ################################


In [ ]:
##############################
# TO DO: Your Implementation
# Use plot_line_segments to recover the pen boundaries. Save the result to
# results/Ques_3/Clutter_Smooth_Image_Output_Edges.png
#
# Suggested approach: first run with min_length=0 and print the longest segments and
# their angles, so you can see what lengths the pens and the clutter actually produce.
# Then pick a band.
raise NotImplementedError('3.6 needs to be implemented.')
##############################
# Your Implementation ENDS here
##############################


#### Your answer to 3.6

*Which pen boundaries did you recover, and what `max_gap`, `min_length` and `max_length` did you use?
If a pen was missed, explain what about it defeated the pipeline - look at its edge map before you
answer.*


## Q4. Circles in a cluttered image (26 points)

### Tasks
1. Smooth the image and comment on the effect - **3 points**
2. Detect edges - **3 points**
3. Run the circle Hough transform - **3 points**
4. Draw the detected circles - **3 points**
5. Explain the false detections - **8 points**
6. Find boundaries for at least 8 coins - **6 points**


### 4.1 Smooth the image for circle detection (3 points)

Save the result as `clutter_smooth_img_circle_detection`. You may use the same sigma as in 3.1 or a
different one; if different, say why in the 4.5 answer.


In [ ]:
clutter_img_circle_detection = cv.imread('../data/proj2-clutter-image.jpg')

##############################
# TO DO: Your Implementation
# Store the smoothed image in clutter_smooth_img_circle_detection.
raise NotImplementedError('4.1 needs to be implemented.')
##############################
# Your Implementation ENDS here
##############################

save_and_show(f'{RESULTS}/Ques_4/proj2-clutter-image-smoothed_circle_detection.png',
              clutter_smooth_img_circle_detection, 'smoothed for circle detection')


### 4.2 Detect edges (3 points)

Save the result as `clutter_smooth_img_gray_edge_circle_detection`. Grayscale first.


In [ ]:
##############################
# TO DO: Your Implementation
# Store the edges in clutter_smooth_img_gray_edge_circle_detection.
raise NotImplementedError('4.2 needs to be implemented.')
##############################
# Your Implementation ENDS here
##############################

save_and_show(f'{RESULTS}/Ques_4/Smooth_Clutter_Edge_Image_Circle_Detection.png',
              clutter_smooth_img_gray_edge_circle_detection,
              'edges for circle detection')


### 4.3 Run the circle Hough transform (3 points)

The coins in this image are not the same size as in Q2 - check the radius range before you run.


In [ ]:
##############################
# TO DO: Your Implementation
# Run circle_hough_transform. Name the outputs clutter_smooth_circle_accumulator,
# clutter_smooth_circle_radius_vector, clutter_smooth_circle_center_x_vector,
# clutter_smooth_circle_center_y_vector, clutter_smooth_circle_img_bin, and save the
# binary image to results/Ques_4/Smooth_Clutter_Binary_Image_Circle_Detection.png
raise NotImplementedError('4.3 needs to be implemented.')
##############################
# Your Implementation ENDS here
##############################


### 4.4 Draw the detected circles (3 points)

In [ ]:
##############################
# TO DO: Your Implementation
# Draw the circles on the smoothed image and save to
# results/Ques_4/Smooth_Clutter_Image_Output_Circles.png
raise NotImplementedError('4.4 needs to be implemented.')
##############################
# Your Implementation ENDS here
##############################


### 4.5 Explain the false detections (8 points)

Did you get circles where there are no coins? Answer in the cell below: what is producing them,
which parameters control them, and what would you change to suppress them without losing real coins?

Look closely at *where* the false positives land before you answer - their location is the clue.


#### Your answer to 4.5

*Where do the false circles appear and what is generating the votes? Which parameters most affect them (binarization threshold, radius range, accumulator threshold, minimum center distance) and in which direction? What is the trade-off when you tighten each one? Propose a concrete change and say what it would cost (which circles will it remove).*


### 4.6 Find at least 8 coins (6 points)

Tune the pipeline until at least 8 coins are correctly circled, and report the count you actually achieved along with the parameters. "Correctly" means the drawn circle sits on a real coin at roughly the right radius - count them yourself from the figure and say how you counted.


In [ ]:
##############################
# TO DO: Your Implementation
# Tune until at least 8 coins are correctly circled. Save the result to
# results/Ques_4/Smooth_Clutter_Image_Circle_Detection_Optimized_4p6.png
raise NotImplementedError('4.6 needs to be implemented.')
##############################
# Your Implementation ENDS here
##############################


#### Your answer to 4.6

*Report the final parameter values, the number of coins correctly detected, the number of false positives, and which coins were missed. Briefly explain why the missed coins may not have been detected.*


## Q5: AI-Use Disclosure (required)

See the syllabus for the details of the course AI policy. 

Answer the questions below. If you didn't use any AI tools, write "None"
for the first question, skip the second. 


### Your disclosure

**1. Which AI tools did you use, if any?**
*(Name the model, the version, the subscription plan, code editor, and any related AI tools. Write "None" if you used none.)*


**2. What did you use them for, and which parts of this notebook did they influence?**
*Be specific about both.
List the cells, functions, or written answers involved.*

---

**Statement.** *By submitting this notebook you confirm that the code and written answers in it are your
own work, that I did not use AI tools to solve any part of this assignment, and that any other AI
use is fully described above.*


---

**Before you submit:** restart the kernel, run every cell from top to bottom, confirm that every
figure and printed result is visible, save the notebook, then run
`python zip_submission.py --uid <your_uid>` from the project folder.
